# AI.FORECAST — BigQuery AI Functions

`AI.FORECAST` is a table-valued function that forecasts future time series values using BigQuery ML's built-in TimesFM model. No model creation or training required.

**When to use it:**
- You want to forecast future values from historical time series data
- You need prediction intervals (confidence bounds) for forecasts
- You want to forecast multiple independent time series at once
- You need a zero-setup forecasting solution (no CREATE MODEL)

**Alternatives:**
- [`AI.DETECT_ANOMALIES`](../ai_detect_anomalies/) — Detect anomalies by comparing data against a forecast baseline
- [`AI.EVALUATE`](../ai_evaluate/) — Evaluate forecast accuracy against actual values

**Featured in:** [Time Series Intelligence](../../workflows/time_series_intelligence/)

**References:** [Full syntax reference](../../RESOURCES.md) | [Official documentation](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-forecast) | [Setup guide](../../setup/)

---
## Setup

Set your project and location, authenticate, and create a temporary dataset for this notebook.

> This function doesn't require a connection or model — it uses end-user credentials automatically. See the [Setup Reference](../../setup/) for details.

In [1]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery dataset location
DATASET_ID = 'bq_ai_functions'  # Shared dataset across all notebooks

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [2]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install('google-cloud-bigquery', 'db-dtypes', 'bigquery-magics', 'tqdm', 'bigframes')

In [3]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [4]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
pd.set_option('display.max_colwidth', None)

# Create the shared dataset (idempotent)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {PROJECT_ID}.{DATASET_ID} ready')

# Register %%bigquery cell magic (auto-loaded in Colab, needed elsewhere)
%load_ext bigquery_magics

Dataset statmike-mlops-349915.bq_ai_functions ready


---
## Examples — SQL

Progressive examples from simplest to most advanced. Each cell adds one new concept.

### Setup: Create sample time series data

Generate synthetic daily sales data with realistic patterns using SQL.

In [5]:
# Create synthetic daily sales data with trend and weekly seasonality
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales` AS
WITH dates AS (
  SELECT date
  FROM UNNEST(GENERATE_DATE_ARRAY('2024-01-01', '2024-12-31')) AS date
)
SELECT
  date,
  -- Base sales + trend + weekly seasonality + noise
  GREATEST(0,
    1000
    + EXTRACT(DAYOFYEAR FROM date) * 2  -- upward trend
    + CASE EXTRACT(DAYOFWEEK FROM date)
        WHEN 1 THEN -200  -- Sunday dip
        WHEN 7 THEN 300   -- Saturday peak
        WHEN 6 THEN 200   -- Friday boost
        ELSE 0
      END
    + CAST(200 * (RAND() - 0.5) AS INT64)  -- random noise
  ) AS daily_sales
FROM dates
'''
client.query(query).result()

# Preview the data
df = client.query(f'SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales` ORDER BY date LIMIT 10').to_dataframe()
print(f'Rows: {client.query(f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales`").to_dataframe().iloc[0]["n"]}')
df

Rows: 366


,date,daily_sales
0,2024-01-01,1100
1,2024-01-02,1095
2,2024-01-03,1052
3,2024-01-04,1101
4,2024-01-05,1110
5,2024-01-06,1264
6,2024-01-07,802
7,2024-01-08,941
8,2024-01-09,997
9,2024-01-10,1060


### 1. Basic forecast

`AI.FORECAST` requires `data_col` and `timestamp_col`. Default horizon is 10 time steps.

In [6]:
query = f'''
SELECT *
FROM AI.FORECAST(
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales`,
  data_col => 'daily_sales',
  timestamp_col => 'date'
)
'''
client.query(query).to_dataframe()

,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,2025-01-01 00:00:00+00:00,1683.175537,0.95,1558.135332,1808.215742,
1,2025-01-02 00:00:00+00:00,1714.604248,0.95,1584.460109,1844.748387,
2,2025-01-03 00:00:00+00:00,1911.007568,0.95,1780.248100,2041.767037,
3,2025-01-04 00:00:00+00:00,2017.196289,0.95,1904.656312,2129.736266,
4,2025-01-05 00:00:00+00:00,1511.278442,0.95,1392.173494,1630.383391,
5,2025-01-06 00:00:00+00:00,1693.976074,0.95,1565.370455,1822.581694,
6,2025-01-07 00:00:00+00:00,1692.562012,0.95,1560.976999,1824.147024,
7,2025-01-08 00:00:00+00:00,1714.832764,0.95,1564.941635,1864.723892,
8,2025-01-09 00:00:00+00:00,1731.646729,0.95,1595.971514,1867.321943,
9,2025-01-10 00:00:00+00:00,1921.637451,0.95,1782.513130,2060.761772,


### 2. Custom horizon

Forecast more time steps into the future.

In [7]:
query = f'''
SELECT *
FROM AI.FORECAST(
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales`,
  data_col => 'daily_sales',
  timestamp_col => 'date',
  horizon => 30
)
'''
client.query(query).to_dataframe()

,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,2025-01-01 00:00:00+00:00,1683.175537,0.95,1558.135332,1808.215742,
1,2025-01-02 00:00:00+00:00,1714.604248,0.95,1584.460109,1844.748387,
2,2025-01-03 00:00:00+00:00,1911.007568,0.95,1780.248100,2041.767037,
3,2025-01-04 00:00:00+00:00,2017.196289,0.95,1904.656312,2129.736266,
4,2025-01-05 00:00:00+00:00,1511.278442,0.95,1392.173494,1630.383391,
5,2025-01-06 00:00:00+00:00,1693.976074,0.95,1565.370455,1822.581694,
6,2025-01-07 00:00:00+00:00,1692.562012,0.95,1560.976999,1824.147024,
7,2025-01-08 00:00:00+00:00,1714.832764,0.95,1564.941635,1864.723892,
8,2025-01-09 00:00:00+00:00,1731.646729,0.95,1595.971514,1867.321943,
9,2025-01-10 00:00:00+00:00,1921.637451,0.95,1782.513130,2060.761772,


### 3. Include historical data

Set `output_historical_time_series => TRUE` to get historical + forecasted data in one result for comparison.

In [8]:
query = f'''
SELECT *
FROM AI.FORECAST(
  (SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales`
   WHERE date >= '2024-11-01'),  -- last 2 months for context
  data_col => 'daily_sales',
  timestamp_col => 'date',
  horizon => 14,
  output_historical_time_series => TRUE
)
ORDER BY time_series_timestamp
'''
df = client.query(query).to_dataframe()
print(f'Historical rows: {(df["time_series_type"] == "history").sum()}')
print(f'Forecast rows: {(df["time_series_type"] == "forecast").sum()}')
df.tail(20)

Historical rows: 61
Forecast rows: 14


,time_series_type,time_series_timestamp,time_series_data,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
55,history,2024-12-26 00:00:00+00:00,1657.000000,NaN,NaN,NaN,None
56,history,2024-12-27 00:00:00+00:00,1897.000000,NaN,NaN,NaN,None
57,history,2024-12-28 00:00:00+00:00,2051.000000,NaN,NaN,NaN,None
58,history,2024-12-29 00:00:00+00:00,1600.000000,NaN,NaN,NaN,None
59,history,2024-12-30 00:00:00+00:00,1631.000000,NaN,NaN,NaN,None
60,history,2024-12-31 00:00:00+00:00,1683.000000,NaN,NaN,NaN,None
61,forecast,2025-01-01 00:00:00+00:00,1694.334229,0.95,1570.239979,1818.428478,
62,forecast,2025-01-02 00:00:00+00:00,1702.731201,0.95,1571.239722,1834.222681,
63,forecast,2025-01-03 00:00:00+00:00,1924.490356,0.95,1824.265037,2024.715676,
64,forecast,2025-01-04 00:00:00+00:00,2008.728394,0.95,1915.322880,2102.133907,


### 4. Multi-series forecasting with id_cols

Forecast multiple independent time series at once by specifying `id_cols`.

In [9]:
# Create multi-series data
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.ai_forecast_multi_series` AS
WITH dates AS (
  SELECT date
  FROM UNNEST(GENERATE_DATE_ARRAY('2024-01-01', '2024-12-31')) AS date
),
stores AS (
  SELECT * FROM UNNEST([
    STRUCT('downtown' AS store_id, 1500 AS base_sales),
    STRUCT('suburban', 800),
    STRUCT('airport', 2000)
  ])
)
SELECT
  s.store_id,
  d.date,
  GREATEST(0, s.base_sales + EXTRACT(DAYOFYEAR FROM d.date) * 1 + CAST(100 * (RAND() - 0.5) AS INT64)) AS daily_sales
FROM dates d CROSS JOIN stores s
'''
client.query(query).result()
print('Multi-series data created')

Multi-series data created


In [10]:
# Forecast each store independently
query = f'''
SELECT *
FROM AI.FORECAST(
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_forecast_multi_series`,
  data_col => 'daily_sales',
  timestamp_col => 'date',
  id_cols => ['store_id'],
  horizon => 14
)
ORDER BY store_id, forecast_timestamp
'''
client.query(query).to_dataframe()

,store_id,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,airport,2025-01-01 00:00:00+00:00,2369.139404,0.95,2316.575967,2421.702841,
1,airport,2025-01-02 00:00:00+00:00,2365.496338,0.95,2310.729952,2420.262723,
2,airport,2025-01-03 00:00:00+00:00,2362.883789,0.95,2306.102643,2419.664935,
3,airport,2025-01-04 00:00:00+00:00,2372.178223,0.95,2314.221299,2430.135146,
4,airport,2025-01-05 00:00:00+00:00,2367.224121,0.95,2303.157193,2431.291049,
5,airport,2025-01-06 00:00:00+00:00,2362.586426,0.95,2301.779860,2423.392992,
6,airport,2025-01-07 00:00:00+00:00,2360.837891,0.95,2295.097841,2426.577941,
7,airport,2025-01-08 00:00:00+00:00,2365.100342,0.95,2295.734765,2434.465918,
8,airport,2025-01-09 00:00:00+00:00,2359.522461,0.95,2290.739736,2428.305186,
9,airport,2025-01-10 00:00:00+00:00,2365.301270,0.95,2299.320762,2431.281777,


### 5. Confidence level and TimesFM model version

Control prediction interval width and model version.

In [11]:
query = f'''
SELECT *
FROM AI.FORECAST(
  TABLE `{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales`,
  data_col => 'daily_sales',
  timestamp_col => 'date',
  horizon => 14,
  confidence_level => 0.99,
  model => 'TimesFM 2.0'
)
'''
client.query(query).to_dataframe()

,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,2025-01-01 00:00:00+00:00,1683.175537,0.99,1518.844852,1847.506223,
1,2025-01-02 00:00:00+00:00,1714.604248,0.99,1543.565857,1885.642639,
2,2025-01-03 00:00:00+00:00,1911.007568,0.99,1739.160497,2082.854640,
3,2025-01-04 00:00:00+00:00,2017.196289,0.99,1869.293688,2165.098890,
4,2025-01-05 00:00:00+00:00,1511.278442,0.99,1354.748006,1667.808878,
5,2025-01-06 00:00:00+00:00,1693.976074,0.99,1524.959640,1862.992508,
6,2025-01-07 00:00:00+00:00,1692.562012,0.99,1519.629992,1865.494032,
7,2025-01-08 00:00:00+00:00,1714.832764,0.99,1517.842429,1911.823099,
8,2025-01-09 00:00:00+00:00,1731.646729,0.99,1553.339272,1909.954185,
9,2025-01-10 00:00:00+00:00,1921.637451,0.99,1738.797100,2104.477803,


---
## Examples — `%%bigquery` Magics

The same examples using IPython magic commands. Magics let you write SQL directly in notebook cells without Python string wrapping.

Key patterns:
- `%%bigquery` — run SQL, display results inline
- `%%bigquery df` — run SQL, capture results into a pandas DataFrame

### Basic forecast with `%%bigquery`

In [12]:
%%bigquery --project {PROJECT_ID}

SELECT *
FROM AI.FORECAST(
  TABLE `statmike-mlops-349915.bq_ai_functions.ai_forecast_sales`,
  data_col => 'daily_sales',
  timestamp_col => 'date',
  horizon => 7
)

Query is running:   0%|          |

Downloading:   0%|          |

,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,2025-01-01 00:00:00+00:00,1683.175537,0.95,1558.135332,1808.215742,
1,2025-01-02 00:00:00+00:00,1714.604248,0.95,1584.460109,1844.748387,
2,2025-01-03 00:00:00+00:00,1911.007568,0.95,1780.248100,2041.767037,
3,2025-01-04 00:00:00+00:00,2017.196289,0.95,1904.656312,2129.736266,
4,2025-01-05 00:00:00+00:00,1511.278442,0.95,1392.173494,1630.383391,
5,2025-01-06 00:00:00+00:00,1693.976074,0.95,1565.370455,1822.581694,
6,2025-01-07 00:00:00+00:00,1692.562012,0.95,1560.976999,1824.147024,


---
## Examples — BigFrames

BigFrames wraps `AI.FORECAST` via `bbq.ai.forecast()`. No model object needed.

In [13]:
import bigframes.pandas as bpd
import bigframes.bigquery as bbq

bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = LOCATION

### Basic forecast

In [14]:
# Load the time series data
df = bpd.read_gbq(f'{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales')

# Forecast
forecast = bbq.ai.forecast(
    df,
    data_col='daily_sales',
    timestamp_col='date',
    horizon=14
)
forecast.to_pandas()

,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,2025-01-12 00:00:00+00:00,1530.235718,0.95,1384.207769,1676.263666,
1,2025-01-10 00:00:00+00:00,1921.637451,0.95,1782.51313,2060.761772,
2,2025-01-08 00:00:00+00:00,1714.832764,0.95,1564.941635,1864.723892,
3,2025-01-02 00:00:00+00:00,1714.604248,0.95,1584.460109,1844.748387,
4,2025-01-13 00:00:00+00:00,1713.45459,0.95,1573.280694,1853.628486,
5,2025-01-01 00:00:00+00:00,1683.175537,0.95,1558.135332,1808.215742,
6,2025-01-05 00:00:00+00:00,1511.278442,0.95,1392.173494,1630.383391,
7,2025-01-11 00:00:00+00:00,2034.317627,0.95,1883.646876,2184.988378,
8,2025-01-07 00:00:00+00:00,1692.562012,0.95,1560.976999,1824.147024,
9,2025-01-06 00:00:00+00:00,1693.976074,0.95,1565.370455,1822.581694,


---
## Cleanup

Drop resources created by this notebook. Shared resources (dataset, models, connection) are left for other notebooks.

In [ ]:
client.delete_table(f'{PROJECT_ID}.{DATASET_ID}.ai_forecast_sales', not_found_ok=True)
print('Table ai_forecast_sales deleted')
client.delete_table(f'{PROJECT_ID}.{DATASET_ID}.ai_forecast_multi_series', not_found_ok=True)
print('Table ai_forecast_multi_series deleted')


### Remove all shared resources

When finished with **all** notebooks, uncomment and run the cell below to delete the shared dataset and all tables, models, and other resources within it.

In [ ]:
# client.delete_dataset(
#     f'{PROJECT_ID}.{DATASET_ID}',
#     delete_contents=True,
#     not_found_ok=True
# )
# print(f'Dataset {PROJECT_ID}.{DATASET_ID} deleted')